In [17]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import classification_report, confusion_matrix

In [18]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [19]:
df.info()

print("\nMissing values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [20]:
df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df = df.dropna()

print("Sau khi xử lý:", df.shape)

Sau khi xử lý: (7032, 20)


In [21]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df = pd.get_dummies(df, drop_first=True)

df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [22]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [25]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)

In [26]:
print("===== RANDOM FOREST =====")
print(classification_report(y_test, y_pred_rf))

print("\n===== SVM =====")
print(classification_report(y_test, y_pred_svm))

===== RANDOM FOREST =====
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.63      0.48      0.54       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.70      1407
weighted avg       0.77      0.79      0.78      1407


===== SVM =====
              precision    recall  f1-score   support

           0       0.82      0.89      0.86      1033
           1       0.62      0.47      0.53       374

    accuracy                           0.78      1407
   macro avg       0.72      0.68      0.69      1407
weighted avg       0.77      0.78      0.77      1407



In [27]:
print("Confusion Matrix - RF")
print(confusion_matrix(y_test, y_pred_rf))

print("\nConfusion Matrix - SVM")
print(confusion_matrix(y_test, y_pred_svm))

Confusion Matrix - RF
[[927 106]
 [196 178]]

Confusion Matrix - SVM
[[924 109]
 [199 175]]


In [28]:
# Lấy classification report dạng dict
rf_report = classification_report(y_test, y_pred_rf, output_dict=True)
svm_report = classification_report(y_test, y_pred_svm, output_dict=True)

# Convert sang DataFrame
rf_df = pd.DataFrame(rf_report).transpose()
svm_df = pd.DataFrame(svm_report).transpose()

# Thêm tên model
rf_df["Model"] = "Random Forest"
svm_df["Model"] = "SVM"

# Gộp lại
final_df = pd.concat([rf_df, svm_df]).reset_index()
final_df.rename(columns={"index": "Class"}, inplace=True)

final_df

,Class,precision,recall,f1-score,support,Model
0,0,0.825467,0.897386,0.859926,1033.000000,Random Forest
1,1,0.626761,0.475936,0.541033,374.000000,Random Forest
2,accuracy,0.785359,0.785359,0.785359,0.785359,Random Forest
3,macro avg,0.726114,0.686661,0.700480,1407.000000,Random Forest
4,weighted avg,0.772648,0.785359,0.775160,1407.000000,Random Forest
5,0,0.822796,0.894482,0.857143,1033.000000,SVM
6,1,0.616197,0.467914,0.531915,374.000000,SVM
7,accuracy,0.781095,0.781095,0.781095,0.781095,SVM
8,macro avg,0.719497,0.681198,0.694529,1407.000000,SVM
9,weighted avg,0.767879,0.781095,0.770693,1407.000000,SVM


In [29]:
# Nếu lỗi thì chạy: pip install openpyxl

with pd.ExcelWriter("model_full_evaluation.xlsx") as writer:
    rf_df.to_excel(writer, sheet_name="Random Forest")
    svm_df.to_excel(writer, sheet_name="SVM")
    final_df.to_excel(writer, sheet_name="All Models", index=False)

print("Đã xuất file model_full_evaluation.xlsx")

Đã xuất file model_full_evaluation.xlsx


In [30]:
os.makedirs("models", exist_ok=True)

joblib.dump(rf, "models/random_forest_model.pkl")
joblib.dump(svm, "models/svm_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")

print("Đã lưu model!")

Đã lưu model!


In [31]:
rf_loaded = joblib.load("models/random_forest_model.pkl")
svm_loaded = joblib.load("models/svm_model.pkl")
scaler_loaded = joblib.load("models/scaler.pkl")

y_pred_rf_loaded = rf_loaded.predict(X_test)

X_test_scaled_loaded = scaler_loaded.transform(X_test)
y_pred_svm_loaded = svm_loaded.predict(X_test_scaled_loaded)

print("Load model OK!")

Load model OK!


In [ ]:
# ================= PHÂN TÍCH KẾT QUẢ =================

# 1. Tổng quan
# Hai mô hình Random Forest và SVM có hiệu năng tương đương,
# trong đó Random Forest nhỉnh hơn một chút.
# Tuy nhiên, sự khác biệt không đáng kể.

# -----------------------------------------------------

# 2. Phân tích Precision
# Precision phản ánh độ chính xác của các dự đoán churn.
# Kết quả cho thấy Precision của lớp churn ở mức trung bình (~0.62).

# Điều này có nghĩa:
# - Khi mô hình dự đoán khách hàng sẽ rời bỏ, khoảng 62% là đúng
# - Khoảng 38% là dự đoán sai

# Trong thực tế:
# - Có thể gây lãng phí chi phí giữ chân khách hàng không cần thiết

# -----------------------------------------------------

# 3. Phân tích Recall (QUAN TRỌNG NHẤT)
# Recall của lớp churn chỉ khoảng ~0.47

# Điều này cho thấy:
# - Mô hình chỉ phát hiện được chưa đến 50% khách hàng thực sự rời bỏ
# - Hơn một nửa khách churn bị bỏ sót

# Trong thực tế:
# - Đây là vấn đề nghiêm trọng vì doanh nghiệp sẽ mất khách mà không biết

# -----------------------------------------------------

# 4. Phân tích F1-score
# F1-score của lớp churn khoảng ~0.53

# Điều này cho thấy:
# - Mô hình chưa cân bằng tốt giữa Precision và Recall
# - Hiệu quả tổng thể còn hạn chế

# -----------------------------------------------------

# 5. Nhận xét về mất cân bằng dữ liệu
# Lớp không churn có kết quả rất cao
# Lớp churn có kết quả thấp hơn đáng kể

# Điều này cho thấy:
# - Dữ liệu bị mất cân bằng
# - Mô hình thiên về lớp chiếm đa số

# -----------------------------------------------------

# 6. Kết luận
# - Random Forest tốt hơn một chút so với SVM
# - Tuy nhiên cả hai mô hình đều chưa tốt trong việc phát hiện churn
# - Recall thấp là điểm yếu lớn nhất

# -----------------------------------------------------

# 7. Hướng cải thiện
# - Sử dụng SMOTE hoặc oversampling
# - Điều chỉnh class_weight
# - Tuning hyperparameters